# Self-Hosted Qwen3-Coder via vLLM on Kaggle (2x NVIDIA T4)
## Relay Gateway Infrastructure Runbook

**Purpose**: This notebook provides an automated, reproducible deployment workflow to launch the self-hosted **Qwen3-Coder-30B-A3B-Instruct** model on Kaggle using **vLLM 0.29.0** and expose it to the **Relay** LLM gateway through an encrypted **Cloudflare Quick Tunnel**.

### System Architecture
```text
Developer / Relay (Local Machine)
        │
        │ HTTPS Request (OpenAI-compatible)
        ▼
Cloudflare Quick Tunnel (*.trycloudflare.com)
        │
        │ Reverse Proxy via Outbound TLS Tunnel
        ▼
Kaggle Dual-T4 Environment
        │
        │ Loopback (http://127.0.0.1:8000/v1)
        ▼
vLLM 0.29.0 Engine (TP=2, FP16, AWQ)
        │
        │ Tensor Parallel Execution
        ▼
QuantTrio/Qwen3-Coder-30B-A3B-Instruct-AWQ (30.5B MoE, ~3.3B active)
```

### Verified Hardware & Runtime Specifications
- **Accelerator**: `GPU T4 x2` (Two NVIDIA Tesla T4 GPUs, ~15,109 MiB usable VRAM each).
- **RAM**: 30 GB container system memory.
- **Model Target**: `QuantTrio/Qwen3-Coder-30B-A3B-Instruct-AWQ` (AWQ 4-bit weights occupy ~15.2 GB total, ~7.6 GB per GPU).
- **Served Model Alias**: `qwen3-coder-30b` (Single alias string; avoids compound-name parsing bugs).
- **vLLM Command**: `vllm serve` (Never use `vllm server`).
- **Flags**: `--tensor-parallel-size 2 --dtype float16 --quantization awq --max-model-len 4096 --max-num-seqs 4 --gpu-memory-utilization 0.85 --enforce-eager --trust-remote-code`
- **Notice**: `--swap-space` is unsupported in vLLM 0.29.0 and omitted.

> [!IMPORTANT]
> **Ephemeral Dev/Testing Environment**: Kaggle sessions are ephemeral (terminated after 9-12 hours or 60 minutes idle). This notebook is intended for development, integration testing, and evaluation of Relay's multi-provider routing and fallback capabilities. It is not persistent production infrastructure.

### Orchestration Model: GitHub -> Kaggle Scripts
This notebook **orchestrates the standalone infrastructure scripts** stored in the Relay repository (`infra/kaggle/`):
- `infra/kaggle/qwen-vllm.sh`: Manages vLLM daemon lifecycle, GPU VRAM checks, local readiness, and safe cleanup.
- `infra/kaggle/cloudflared.sh`: Manages `cloudflared` binary installation, background tunnel daemon, and dynamic URL discovery.
- `infra/kaggle/diagnostics.sh`: 10-point health inspection suite covering GPU, CUDA, port, process, tunnel, and network status.

Let's begin the step-by-step deployment!


---
### Cell 1: Kaggle Environment Verification
- **Purpose**: Inspect the Python runtime, Linux OS platform, container user, and default working directory.
- **Where**: Kaggle Jupyter Notebook container.
- **How**: Queries standard Python `sys`, `platform`, `os`, and system memory info.
- **Expected Result**: Linux x86_64, Python 3.10+, working directory `/kaggle/working`.
- **What Failure Means**: Non-standard container environment.
- **What to Do Next**: Proceed to GPU check.


In [ ]:
import os
import sys
import platform

print("=" * 60)
print("CELL 1: Kaggle Environment Information")
print("=" * 60)
print(f"Python Executable: {sys.executable}")
print(f"Python Version:    {sys.version.split()[0]}")
print(f"Platform:          {platform.platform()}")
print(f"System Processor:  {platform.processor() or 'x86_64'}")
print(f"Working Directory: {os.getcwd()}")

# Inspect available system RAM
try:
    with open("/proc/meminfo", "r") as f:
        for line in f:
            if "MemTotal" in line or "MemAvailable" in line:
                print(f"RAM:               {line.strip()}")
except Exception as e:
    print(f"Could not read /proc/meminfo: {e}")

print("=" * 60)
print("STATUS: Environment check PASS. Ready for GPU inspection.")


---
### Cell 2: GPU Hardware & VRAM Check
- **Purpose**: Verify that dual NVIDIA Tesla T4 GPUs are allocated and visible with clean memory before launching vLLM.
- **Where**: Kaggle GPU host.
- **How**: Invokes `nvidia-smi` and checks PyTorch CUDA visibility (`torch.cuda.device_count()`).
- **Canonical Verified Environment**: 2 × NVIDIA Tesla T4 (~15,109 MiB free VRAM each).
- **Expected Result**: Exactly 2 GPUs detected, model reported as `Tesla T4`, ~0 MiB used.
- **What Failure Means**: Accelerator is set to CPU or single GPU, or a prior process is holding VRAM.
- **What to Do Next**: If fewer than 2 GPUs, open right sidebar -> **Notebook settings** -> **Accelerator** -> **GPU T4 x2**.


In [ ]:
import subprocess
import shutil

print("=" * 60)
print("CELL 2: GPU & CUDA Device Check")
print("=" * 60)

if not shutil.which("nvidia-smi"):
    print("[FAIL] 'nvidia-smi' not found! Ensure Accelerator is set to 'GPU T4 x2' in Kaggle settings.")
else:
    # Run nvidia-smi summary
    subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,driver_version", "--format=csv"], check=False)
    
    # Check with PyTorch
    try:
        import torch
        cuda_avail = torch.cuda.is_available()
        dev_count = torch.cuda.device_count()
        print(f"\nPyTorch Version:    {torch.__version__}")
        print(f"CUDA Available:     {cuda_avail}")
        print(f"CUDA Device Count:  {dev_count}")
        
        for i in range(dev_count):
            name = torch.cuda.get_device_name(i)
            free_mb = torch.cuda.mem_get_info(i)[0] / (1024 ** 2)
            total_mb = torch.cuda.mem_get_info(i)[1] / (1024 ** 2)
            print(f"  Device {i}: {name} ({free_mb:.0f} MiB free / {total_mb:.0f} MiB total)")
            
        if dev_count < 2:
            print("\n[WARNING] Fewer than 2 GPUs detected. Tensor parallelism (TP=2) requires 2 GPUs.")
            print("Action: Set Accelerator to 'GPU T4 x2' in Kaggle Notebook Settings.")
        else:
            print("\n[PASS] Dual GPU environment verified. Ready to clone Relay repository.")
    except Exception as e:
        print(f"PyTorch CUDA check error: {e}")


---
### Cell 3: Clone Relay GitHub Repository
- **Purpose**: Pull the version-controlled Relay repository containing the infrastructure management scripts (`infra/kaggle/`) into the Kaggle filesystem.
- **Where**: Kaggle container filesystem (`/kaggle/working`).
- **How**: Clones `https://github.com/mosabbir-maruf/Relay.git` to `/kaggle/working/Relay`. If already cloned, inspects status without overwriting uncommitted work.
- **Where the files appear**:
  - Root: `/kaggle/working/Relay`
  - Scripts: `/kaggle/working/Relay/infra/kaggle/`
- **Expected Result**: Repository cloned, commit hash printed, working directory switched to `/kaggle/working/Relay`.
- **What Failure Means**: Internet is disabled in Kaggle settings or repository URL is unreachable.
- **What to Do Next**: Ensure **Internet: On** in the right sidebar under Notebook settings.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/mosabbir-maruf/Relay.git"
CLONE_DEST = "/kaggle/working/Relay"

print("=" * 60)
print("CELL 3: Clone Relay GitHub Repository")
print("=" * 60)

if not os.path.exists(CLONE_DEST):
    print(f"Cloning {REPO_URL} into {CLONE_DEST}...")
    subprocess.run(["git", "clone", REPO_URL, CLONE_DEST], check=True)
    print("Clone complete.")
else:
    print(f"Repository already exists at {CLONE_DEST}. Inspecting status...")
    subprocess.run(["git", "-C", CLONE_DEST, "status", "--short"], check=False)

# Switch current working directory into repository
os.chdir(CLONE_DEST)
print(f"\nCurrent Working Directory: {os.getcwd()}")

# Print latest Git commit info
commit_info = subprocess.check_output(
    ["git", "log", "-n", "1", "--pretty=format:%h - %s (%ci)"],
    text=True
).strip()
print(f"Active Git Commit:        {commit_info}")
print("=" * 60)
print("STATUS: Repository ready. Proceeding to verify scripts.")


---
### Cell 4: Verify Kaggle Infrastructure Scripts
- **Purpose**: Confirm that all required management scripts exist in `infra/kaggle/` and are readable.
- **Where**: `/kaggle/working/Relay/infra/kaggle/`.
- **How**: Validates file paths:
  1. `infra/kaggle/README.md`
  2. `infra/kaggle/qwen-vllm.sh`
  3. `infra/kaggle/cloudflared.sh`
  4. `infra/kaggle/diagnostics.sh`
- **Expected Result**: All 4 files report `PASS` with valid sizes.
- **What Failure Means**: Incomplete git clone or missing files.
- **What to Do Next**: Proceed to granting execute permissions.


In [ ]:
import os

print("=" * 60)
print("CELL 4: Verify Infrastructure Files")
print("=" * 60)

expected_files = [
    "infra/kaggle/README.md",
    "infra/kaggle/qwen-vllm.sh",
    "infra/kaggle/cloudflared.sh",
    "infra/kaggle/diagnostics.sh"
]

all_passed = True
for rel_path in expected_files:
    full_path = os.path.abspath(rel_path)
    exists = os.path.isfile(full_path)
    size = os.path.getsize(full_path) if exists else 0
    status = "PASS" if exists and size > 0 else "FAIL"
    if status == "FAIL":
        all_passed = False
    print(f"[{status}] {rel_path} ({size:,} bytes)")

print("=" * 60)
if all_passed:
    print("STATUS: All infrastructure scripts verified successfully.")
else:
    raise FileNotFoundError("One or more required infrastructure scripts are missing!")


---
### Cell 5: Make Scripts Executable (`chmod +x`)
- **Purpose**: Apply POSIX execute permissions (`+x`) to the shell scripts.
- **Why this is necessary**: GitHub preserves source code and text content, but depending on the container filesystem mount inside Kaggle, files may not have POSIX execute bits enabled. Applying `chmod +x` ensures Python subprocesses and bash cells can invoke `./infra/kaggle/<script>.sh` directly.
- **Where the files are**: `/kaggle/working/Relay/infra/kaggle/*.sh`.
- **How to run**: `chmod +x infra/kaggle/*.sh`.
- **Expected Result**: All `.sh` files report `executable: True`.
- **What Failure Means**: Filesystem permission error.
- **What to Do Next**: Proceed to preflight check.


In [ ]:
import os
import subprocess
import glob

print("=" * 60)
print("CELL 5: Make Scripts Executable (chmod +x)")
print("=" * 60)

script_patterns = "infra/kaggle/*.sh"
scripts = glob.glob(script_patterns)

for script in sorted(scripts):
    subprocess.run(["chmod", "+x", script], check=True)
    is_exec = os.access(script, os.X_OK)
    print(f"chmod +x {script} -> Executable: {is_exec}")

print("=" * 60)
print("STATUS: All shell scripts are marked executable.")


---
### Cell 6: Qwen / vLLM Preflight Check
- **Purpose**: Verify prerequisite packages (installing `vllm` if needed), inspect CUDA runtime paths, and check port 8000 availability.
- **Where the file is**: `/kaggle/working/Relay/infra/kaggle/qwen-vllm.sh`.
- **How to run**: `./infra/kaggle/qwen-vllm.sh check`.
- **Expected Result**:
  - GPU Hardware: 2 GPUs detected.
  - vLLM CLI: available on PATH.
  - Port 8000: free.
- **What Failure Means**: `vllm` binary missing, port 8000 in use, or fewer than 2 GPUs.
- **What to Do Next**: The cell automatically ensures `vllm` is installed via pip if missing.


In [ ]:
import subprocess
import shutil

print("=" * 60)
print("CELL 6: Qwen / vLLM Preflight Check")
print("=" * 60)

# Check if vllm CLI is present; if not, install cleanly
if not shutil.which("vllm"):
    print("vLLM CLI not detected. Installing vllm package (takes ~60s)...")
    subprocess.run(["pip", "install", "-q", "--no-cache-dir", "vllm"], check=True)
    print("vLLM installation complete.")

# Run the canonical preflight check
subprocess.run(["./infra/kaggle/qwen-vllm.sh", "check"], check=True)

print("=" * 60)
print("STATUS: Preflight check complete.")


---
### Cell 7: System Diagnostics Before Startup
- **Purpose**: Run the comprehensive 10-point diagnostic suite to check GPU memory, PyTorch CUDA device allocation, port 8000 occupancy, and background daemons.
- **Where the file is**: `/kaggle/working/Relay/infra/kaggle/diagnostics.sh`.
- **How to run**: `./infra/kaggle/diagnostics.sh all`.
- **Why**: Essential when starting from an unknown state or recovering from a previous interrupted session to confirm no orphaned processes are occupying GPU VRAM.
- **Expected Result**: Diagnostic report showing GPU VRAM, PyTorch allocation PASS, and port status.
- **What Failure Means**: Stale workers or CUDA configuration issue.
- **What to Do Next**: Proceed to safe cleanup in Cell 8.


In [ ]:
import subprocess

print("=" * 60)
print("CELL 7: Full Stack Pre-Startup Diagnostics")
print("=" * 60)

subprocess.run(["./infra/kaggle/diagnostics.sh", "all"], check=False)

print("=" * 60)
print("STATUS: Diagnostics complete. Proceeding to safe cleanup.")


---
### Cell 8: Safe Process & Port Cleanup
- **Purpose**: Safely detect and terminate any lingering vLLM or Python worker processes bound to port 8000 without blindly killing unrelated Jupyter notebook processes.
- **Where the file is**: `/kaggle/working/Relay/infra/kaggle/qwen-vllm.sh`.
- **How to run**: `./infra/kaggle/qwen-vllm.sh clean`.
- **Safety Guarantee**: The script checks recorded PID in `/kaggle/working/vllm.pid` and inspects process command names (`python`, `vllm`) before issuing termination signals. Confirms GPU VRAM is released.
- **Expected Result**: "No stale vLLM processes detected" or "Cleanup complete", with GPU memory showing 0 MiB used.
- **What to Do Next**: Ready to launch vLLM in Cell 9.


In [ ]:
import subprocess

print("=" * 60)
print("CELL 8: Safe Cleanup of Stale Workers")
print("=" * 60)

subprocess.run(["./infra/kaggle/qwen-vllm.sh", "clean"], check=True)

print("=" * 60)
print("STATUS: Safe cleanup complete. Environment is clean for vLLM startup.")


---
### Cell 9: Start vLLM Server in Background
- **Purpose**: Launch the canonical `vllm serve` daemon in the background with tensor parallelism across both T4 GPUs.
- **Where the file is**: `/kaggle/working/Relay/infra/kaggle/qwen-vllm.sh`.
- **How to run**: `./infra/kaggle/qwen-vllm.sh start`.
- **Canonical Model & Serving Configuration**:
  - **Model**: `QuantTrio/Qwen3-Coder-30B-A3B-Instruct-AWQ`
  - **Served Model Name**: `qwen3-coder-30b`
  - **vLLM Command**: `vllm serve` (NOT `vllm server`)
  - **Tensor Parallel Size**: `2`
  - **DType**: `float16` (Mandatory for Turing T4 architecture)
  - **Quantization**: `awq`
  - **Max Model Length**: `4096`
  - **Max Num Sequences**: `4`
  - **GPU Memory Utilization**: `0.85`
  - **Enforce Eager**: `true` (Disables CUDA graphs, conserving ~1-2 GB VRAM)
  - **Trust Remote Code**: `true`
  - **Host / Port**: `0.0.0.0:8000`
- **Why Background Execution is Mandatory**: Foreground execution blocks the Jupyter notebook cell indefinitely. Launching in the background via `nohup` allows subsequent cells to monitor logs, run health checks, and start the tunnel.
- **Expected Result**: Background PID recorded in `/kaggle/working/vllm.pid`, logs streaming to `/kaggle/working/vllm_server.log`.
- **What if it fails**: If the process terminates immediately, inspect `/kaggle/working/vllm_server.log`.


In [ ]:
import subprocess

print("=" * 60)
print("CELL 9: Launch vLLM Server in Background")
print("=" * 60)

subprocess.run(["./infra/kaggle/qwen-vllm.sh", "start"], check=True)

print("=" * 60)
print("STATUS: vLLM process launched. Proceeding to readiness polling.")


---
### Cell 10: Wait for vLLM Readiness (Polling Loop)
- **Purpose**: Poll `http://127.0.0.1:8000/v1/models` in a loop until the model weights (~15.2 GB) are loaded and the KV cache is compiled.
- **Where**: Kaggle localhost (`http://127.0.0.1:8000`).
- **How**: Polling loop using Python `urllib.request` with a 10-minute timeout (checking every 5 seconds).
- **Status Progression**: `[WAITING]` -> `[READY]` (or `[TIMEOUT]`).
- **Expected Result**: HTTP 200 response within 2-4 minutes indicating server is ready.
- **What Failure Means**: Out of memory (OOM), weight download failure, or process crashed.
- **What to Do Next**: If timeout occurs, inspect `/kaggle/working/vllm_server.log`. Do not start another server repeatedly.


In [ ]:
import time
import urllib.request
import json
import subprocess

HEALTH_URL = "http://127.0.0.1:8000/v1/models"
TIMEOUT_SECONDS = 600  # 10 minutes maximum for initial weight download & cache compile
POLL_INTERVAL = 5

print("=" * 60)
print("CELL 10: Waiting for vLLM Server Readiness")
print(f"Target: {HEALTH_URL} (Timeout: {TIMEOUT_SECONDS}s)")
print("=" * 60)

start_time = time.time()
is_ready = False

while time.time() - start_time < TIMEOUT_SECONDS:
    elapsed = int(time.time() - start_time)
    try:
        req = urllib.request.Request(HEALTH_URL)
        with urllib.request.urlopen(req, timeout=3) as resp:
            if resp.status == 200:
                data = json.loads(resp.read().decode("utf-8"))
                models = [m.get("id") for m in data.get("data", [])]
                print(f"\n[READY] vLLM is ONLINE! (Elapsed: {elapsed}s)")
                print(f"Registered Models: {models}")
                is_ready = True
                break
    except Exception:
        if elapsed % 20 == 0:
            print(f"[WAITING] Loading weights & initializing KV cache ({elapsed}s elapsed)...")
    time.sleep(POLL_INTERVAL)

print("=" * 60)
if not is_ready:
    print("[TIMEOUT] vLLM server did not become ready within timeout!")
    print("Recent server logs (last 40 lines):")
    subprocess.run(["./infra/kaggle/qwen-vllm.sh", "logs", "40"], check=False)
    raise TimeoutError("vLLM server startup timed out. Check logs above for details.")
else:
    print("STATUS: Server is healthy and accepting requests.")


---
### Cell 11: Local Model Discovery (`GET /v1/models`)
- **Purpose**: Verify that the `/v1/models` endpoint exposes the exact single model alias: `qwen3-coder-30b`.
- **Where**: Kaggle localhost (`http://127.0.0.1:8000/v1/models`).
- **How**: Performs HTTP GET and parses JSON model IDs.
- **Expected Result**: Returns HTTP 200 with model ID `qwen3-coder-30b` (not a compound comma string).
- **What Failure Means**: Incorrect `--served-model-name` argument passed during startup.
- **What to Do Next**: Proceed to local inference test.


In [ ]:
import urllib.request
import json

print("=" * 60)
print("CELL 11: Local Model Discovery Test")
print("=" * 60)

url = "http://127.0.0.1:8000/v1/models"
req = urllib.request.Request(url)

with urllib.request.urlopen(req, timeout=5) as resp:
    status_code = resp.status
    payload = json.loads(resp.read().decode("utf-8"))

print(f"HTTP Status: {status_code}")
print(f"Response Body: {json.dumps(payload, indent=2)}")

model_ids = [m["id"] for m in payload.get("data", [])]
assert "qwen3-coder-30b" in model_ids, f"Expected 'qwen3-coder-30b' in {model_ids}"

print("\n[PASS] Model 'qwen3-coder-30b' verified successfully.")


---
### Cell 12: Local Inference Test (PONG Prompt)
- **Purpose**: Execute a non-streaming test inference request against the local vLLM server to confirm GPU token generation is functional.
- **Where the file is**: `/kaggle/working/Relay/infra/kaggle/qwen-vllm.sh`.
- **How to run**: `./infra/kaggle/qwen-vllm.sh test` (or direct HTTP POST).
- **Prompt**: `"Reply with only the word PONG"`.
- **Expected Result**: HTTP 200 response with `"content": "PONG"`.
- **What Failure Means**: Out of memory during forward pass, or CUDA execution failure.
- **What to Do Next**: Proceed to streaming verification.


In [ ]:
import subprocess

print("=" * 60)
print("CELL 12: Local Non-Streaming Inference Test")
print("=" * 60)

# Invoke canonical test via management script
subprocess.run(["./infra/kaggle/qwen-vllm.sh", "test"], check=True)

print("=" * 60)
print("STATUS: Local non-streaming inference PASS.")


---
### Cell 13: Local Streaming Inference Test (SSE)
- **Purpose**: Validate that vLLM Server-Sent Events (SSE) streaming works correctly (`stream=True`) and emits the terminal `data: [DONE]` frame.
- **Where**: Kaggle localhost (`http://127.0.0.1:8000/v1/chat/completions`).
- **How**: Sends HTTP POST with `"stream": True` and parses chunked SSE delta tokens.
- **Expected Result**: Successive streaming chunks received, ending with `data: [DONE]`.
- **What Failure Means**: SSE parsing or chunk buffering problem.
- **What to Do Next**: Proceed to Cloudflare Tunnel setup.


In [ ]:
import urllib.request
import json

print("=" * 60)
print("CELL 13: Local Streaming Inference Test (SSE)")
print("=" * 60)

payload = {
    "model": "qwen3-coder-30b",
    "messages": [{"role": "user", "content": "Count from 1 to 5"}],
    "temperature": 0.0,
    "max_tokens": 30,
    "stream": True
}

req = urllib.request.Request(
    "http://127.0.0.1:8000/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)

received_done = False
collected_text = ""

with urllib.request.urlopen(req, timeout=30) as resp:
    for raw_line in resp:
        line = raw_line.decode("utf-8").strip()
        if not line or line.startswith(":"):
            continue
        if line == "data: [DONE]":
            received_done = True
            break
        if line.startswith("data: "):
            chunk = json.loads(line[6:])
            choices = chunk.get("choices", [])
            if choices:
                delta = choices[0].get("delta", {}).get("content", "")
                collected_text += delta
                print(delta, end="", flush=True)

print(f"\n\n[PASS] Stream completed. Received [DONE]: {received_done}")
assert received_done, "Stream did not terminate with [DONE]"
print("STATUS: Local streaming inference verified.")


---
### Cell 14: cloudflared Binary Verification
- **Purpose**: Verify that the official `cloudflared` Linux binary is installed and executable in `/kaggle/working/cloudflared`.
- **Where the file is**: `/kaggle/working/Relay/infra/kaggle/cloudflared.sh`.
- **How to run**: `./infra/kaggle/cloudflared.sh check`.
- **Automatic Behavior**: If the binary is missing, `cloudflared.sh` automatically downloads the latest official release from Cloudflare's GitHub and marks it executable.
- **Expected Result**: Binary path and version displayed.
- **What Failure Means**: Download failure (check Kaggle internet access).
- **What to Do Next**: Proceed to starting the tunnel.


In [ ]:
import subprocess

print("=" * 60)
print("CELL 14: cloudflared Binary Verification")
print("=" * 60)

subprocess.run(["./infra/kaggle/cloudflared.sh", "check"], check=True)

print("=" * 60)
print("STATUS: cloudflared binary verified. Ready to start tunnel.")


---
### Cell 15: Start Cloudflare Quick Tunnel
- **Purpose**: Launch a Cloudflare Quick Tunnel targeting `http://127.0.0.1:8000` in the background and dynamically extract the assigned public HTTPS URL.
- **Where the file is**: `/kaggle/working/Relay/infra/kaggle/cloudflared.sh`.
- **How to run**: `./infra/kaggle/cloudflared.sh start`.
- **Dynamic Discovery Rule**: Quick Tunnel URLs are randomly assigned (`https://<subdomain>.trycloudflare.com`). **Never hardcode tunnel URLs.** The script dynamically regex-extracts the URL from `/kaggle/working/cloudflared.log`.
- **Expected Result**: Public URL displayed (e.g. `https://random-words.trycloudflare.com`).
- **What Failure Means**: Tunnel process failed to connect to Cloudflare edge network.
- **What to Do Next**: Proceed to testing the public endpoint.


In [ ]:
import subprocess
import time

print("=" * 60)
print("CELL 15: Start Cloudflare Quick Tunnel")
print("=" * 60)

# Start tunnel daemon
subprocess.run(["./infra/kaggle/cloudflared.sh", "start"], check=True)

# Query dynamic public URL
PUBLIC_TUNNEL_URL = ""
for _ in range(15):
    try:
        url_output = subprocess.check_output(["./infra/kaggle/cloudflared.sh", "url"], text=True).strip()
        if "trycloudflare.com" in url_output:
            PUBLIC_TUNNEL_URL = url_output
            break
    except Exception:
        pass
    time.sleep(2)

print("=" * 60)
if PUBLIC_TUNNEL_URL:
    print(f"SUCCESS: Active Public Tunnel URL -> {PUBLIC_TUNNEL_URL}")
else:
    print("WARNING: Could not extract URL yet. Inspecting logs:")
    subprocess.run(["./infra/kaggle/cloudflared.sh", "logs", "25"], check=False)
    raise RuntimeError("Failed to retrieve public Cloudflare Tunnel URL.")


---
### Cell 16: Public Model Discovery Test
- **Purpose**: Verify that the public Cloudflare HTTPS endpoint successfully proxies traffic over the internet to the local vLLM `/v1/models` endpoint.
- **Where**: Internet -> Cloudflare Edge -> Kaggle.
- **How**: Executes HTTP GET against `<PUBLIC_TUNNEL_URL>/v1/models`.
- **Expected Result**: HTTP 200 with model ID `qwen3-coder-30b`.
- **What Failure Means**: Cloudflare tunnel disconnected or DNS propagation delay.
- **What to Do Next**: Proceed to public inference test.


In [ ]:
import urllib.request
import json

print("=" * 60)
print("CELL 16: Public Model Discovery Test")
print(f"Target: {PUBLIC_TUNNEL_URL}/v1/models")
print("=" * 60)

req = urllib.request.Request(f"{PUBLIC_TUNNEL_URL}/v1/models")
with urllib.request.urlopen(req, timeout=15) as resp:
    status_code = resp.status
    payload = json.loads(resp.read().decode("utf-8"))

print(f"HTTP Status: {status_code}")
print(f"Models:      {[m['id'] for m in payload.get('data', [])]}")
assert "qwen3-coder-30b" in [m["id"] for m in payload.get("data", [])]

print("=" * 60)
print("[PASS] Public model discovery verified over HTTPS.")


---
### Cell 17: Public End-to-End Inference Test
- **Purpose**: Execute an end-to-end chat completion request through the public Cloudflare tunnel to confirm external clients (like Relay) can communicate with Qwen.
- **Request Flow**: `Client -> Cloudflare Edge -> Reverse Proxy -> Local vLLM -> Qwen3-Coder -> Token Response`.
- **Prompt**: `"Reply with only the word PONG"`.
- **Expected Result**: HTTP 200 response containing `PONG`.
- **What Failure Means**: Edge timeout, payload size issue, or tunnel disconnection.
- **What to Do Next**: Proceed to generating Relay configuration.


In [ ]:
import urllib.request
import json

print("=" * 60)
print("CELL 17: Public End-to-End Inference Test")
print(f"Target: {PUBLIC_TUNNEL_URL}/v1/chat/completions")
print("=" * 60)

payload = {
    "model": "qwen3-coder-30b",
    "messages": [{"role": "user", "content": "Reply with only the word PONG"}],
    "temperature": 0.0,
    "max_tokens": 16
}

req = urllib.request.Request(
    f"{PUBLIC_TUNNEL_URL}/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)

with urllib.request.urlopen(req, timeout=30) as resp:
    status_code = resp.status
    result = json.loads(resp.read().decode("utf-8"))

content = result["choices"][0]["message"]["content"].strip()
print(f"HTTP Status: {status_code}")
print(f"Response:    {content}")
assert "PONG" in content.upper(), f"Expected PONG, got: {content}"

print("=" * 60)
print("[PASS] End-to-end public inference verified successfully!")


---
### Cell 18: Generate Relay Configuration Snippet
- **Purpose**: Generate the exact environment configuration variables to copy into the local `.env` file on your development machine.
- **Where to use**: On your computer in the root directory of your cloned Relay repository.
- **Important Note**: Quick Tunnel URLs are ephemeral and change whenever `cloudflared` is restarted. **Do not commit these temporary URLs to git.**


In [ ]:
print("=" * 60)
print("CELL 18: Copy-Paste Configuration for Local Relay Gateway")
print("=" * 60)
print("Paste these lines into your local .env file on your computer:\n")
print(f"QWEN_BASE_URL={PUBLIC_TUNNEL_URL}/v1")
print(f"QWEN_MODEL=qwen3-coder-30b")
print("\n" + "=" * 60)
print("HOW TO RUN RELAY LOCALLY:")
print("1. In your local Relay repository: pnpm dev")
print("2. Test routing through Relay:")
print('   curl -X POST http://localhost:3000/v1/chat/completions \\')
print('     -H "Content-Type: application/json" \\')
print('     -d '{"model": "qwen3-coder-30b", "messages": [{"role": "user", "content": "Hello!"}]}'')
print("=" * 60)


---
### Cell 19: Operational Status & Resource Monitoring
- **Purpose**: Inspect the live status of all components (vLLM daemon, cloudflared tunnel, GPU memory allocation, and endpoints).
- **Where the file is**: `/kaggle/working/Relay/infra/kaggle/diagnostics.sh`.
- **How to run**: `./infra/kaggle/diagnostics.sh tunnel` and `./infra/kaggle/qwen-vllm.sh status`.
- **Expected Result**: vLLM RUNNING, cloudflared RUNNING, GPU VRAM ~8-9 GB per GPU, public URL active.


In [ ]:
import subprocess

print("=" * 60)
print("CELL 19: Operational Stack Summary")
print("=" * 60)

# Check vLLM status
subprocess.run(["./infra/kaggle/qwen-vllm.sh", "status"], check=False)

# Check tunnel status
print("\n" + "-" * 40)
subprocess.run(["./infra/kaggle/cloudflared.sh", "status"], check=False)

# Check GPU memory
print("\n" + "-" * 40)
print("Current GPU VRAM Allocation:")
subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"], check=False)

print("=" * 60)
print("STATUS: Backend is fully operational and serving traffic.")


---
### Cell 20: Safe Shutdown Procedure
- **Purpose**: Cleanly terminate both the Cloudflare Quick Tunnel and the vLLM server processes, release all GPU VRAM, and clean up PID files when you are finished testing.
- **Where the files are**:
  - `/kaggle/working/Relay/infra/kaggle/cloudflared.sh`
  - `/kaggle/working/Relay/infra/kaggle/qwen-vllm.sh`
- **How to run**:
  1. `./infra/kaggle/cloudflared.sh stop`
  2. `./infra/kaggle/qwen-vllm.sh stop`
- **Expected Result**: Both daemons stopped, GPU VRAM cleared back to 0 MiB.
- **Next Step**: You can now safely stop or close the Kaggle session.


In [ ]:
import subprocess
import time

print("=" * 60)
print("CELL 20: Safe Shutdown & Resource Cleanup")
print("=" * 60)

# 1. Stop Cloudflare Tunnel
print("1. Stopping Cloudflare Tunnel...")
subprocess.run(["./infra/kaggle/cloudflared.sh", "stop"], check=False)

# 2. Stop vLLM Server
print("\n2. Stopping vLLM Server...")
subprocess.run(["./infra/kaggle/qwen-vllm.sh", "stop"], check=False)

# 3. Verify GPU memory is freed
print("\n3. Verifying GPU Memory Release:")
time.sleep(3)
subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.free", "--format=csv"], check=False)

print("=" * 60)
print("SUCCESS: All processes terminated cleanly. GPU memory released.")
print("The Kaggle session can now be safely closed.")


---
## Troubleshooting & Failure Recovery Reference

This reference covers the real-world operational issues encountered during this project and their verified remediations:

### A. vLLM CLI Error: `unrecognized subcommand 'server'`
- **Cause**: Using the obsolete command `vllm server`.
- **Remediation**: Use `vllm serve`.

### B. Unsupported Argument: `--swap-space`
- **Cause**: `--swap-space` was deprecated/removed in vLLM 0.29.0 on Kaggle.
- **Remediation**: Omit `--swap-space`. Manage memory headroom via `--gpu-memory-utilization 0.85`.

### C. Served Model Naming Issue
- **Cause**: Passing comma-separated names (`QuantTrio/...,qwen3-coder-30b`) causes vLLM to treat it as a literal compound name.
- **Remediation**: Pass a single string: `--served-model-name qwen3-coder-30b`.

### D. Stale vLLM Worker Processes / High VRAM on Startup
- **Symptoms**: `torch.cuda.OutOfMemoryError` or `WorkerProc failed to start` when restarting.
- **Remediation**:
  1. Run `./infra/kaggle/qwen-vllm.sh clean`
  2. Confirm with `nvidia-smi` that VRAM used is ~0 MiB.
  3. Restart vLLM: `./infra/kaggle/qwen-vllm.sh start`

### E. WorkerProc Failed to Start
- **Causes**: Port 8000 occupied, insufficient VRAM, or missing CUDA 13 libraries.
- **Remediation**: Run `./infra/kaggle/diagnostics.sh all`. Do not restart blindly in a loop.

### F. Port 8000 Occupied
- **Cause**: Previous vLLM instance still listening on port 8000.
- **Remediation**: Run `./infra/kaggle/qwen-vllm.sh clean`.

### G. Server Running but `/v1/models` Not Ready
- **Cause**: Model weights (~15.2 GB) are still downloading from Hugging Face or the KV cache is compiling.
- **Remediation**: Inspect logs: `tail -n 100 /kaggle/working/vllm_server.log` or `./infra/kaggle/qwen-vllm.sh logs 50`. Initial compilation takes 2-4 minutes.

### H. `cloudflared` Binary Missing
- **Cause**: Binary has not been downloaded to the Kaggle session.
- **Remediation**: Run `./infra/kaggle/cloudflared.sh check` to automatically download the official binary.

### I. Tunnel URL Not Detected
- **Cause**: Cloudflare edge network latency or internet disabled in Kaggle.
- **Remediation**: Check logs: `./infra/kaggle/cloudflared.sh logs 30`. Verify **Internet: On** in Kaggle settings.

### J. Public Endpoint Fails (6-Step Isolation Order)
When requests to the public URL fail, test in this exact sequential order to isolate the fault layer:
1. `curl http://127.0.0.1:8000/v1/models`: If fails -> vLLM crashed. Check `vllm_server.log`.
2. `./infra/kaggle/qwen-vllm.sh test`: If fails -> Forward pass OOM or engine error.
3. `./infra/kaggle/cloudflared.sh status`: If fails -> `cloudflared` died.
4. `./infra/kaggle/cloudflared.sh logs`: If errors -> Network or edge tunnel dropped.
5. `curl https://<subdomain>.trycloudflare.com/v1/models`: If fails -> DNS/edge propagation.
6. `curl -X POST https://<subdomain>.trycloudflare.com/v1/chat/completions ...`: If fails -> Payload/timeout issue.
